In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [3]:
df = pd.read_csv(
    INTERIM_DIR / "merged_air_quality.csv",
    parse_dates=['time'],
    index_col='time'
)

In [4]:
df.head()

,pm10,pm25,no2,so2,co,temperature,rain,wind_direction,wind_speed,humidity,pressure,dew_point
time,,,,,,,,,,,,
2022-09-01 00:00:00,53.1,37.1,29.3,5.0,765,20.6,1.2,243,0.8,95,870.2,19.8
2022-09-01 01:00:00,51.6,36.2,24.5,5.4,728,20.7,8.4,194,1.5,95,870.9,19.8
2022-09-01 02:00:00,50.6,35.5,18.8,5.8,679,20.9,9.5,225,1.5,94,871.7,19.9
2022-09-01 03:00:00,49.6,34.8,15.9,6.0,653,21.1,7.2,207,0.8,94,872.4,20.2
2022-09-01 04:00:00,47.1,33.1,13.9,6.1,629,21.4,2.6,45,1.0,93,873.0,20.3


In [5]:
print("Shape:", df.shape)

print(
    "Period:",
    df.index.min(),
    "to",
    df.index.max()
)

print(
    "Frequency:",
    pd.infer_freq(df.index)
)

print(
    "Duplicate timestamps:",
    df.index.duplicated().sum()
)

print(
    "Missing values:",
    df.isna().sum().sum()
)

Shape: (23352, 12)
Period: 2022-09-01 00:00:00 to 2025-04-30 23:00:00
Frequency: h
Duplicate timestamps: 0
Missing values: 0


In [6]:
expected_columns = [
    'pm10',
    'pm25',
    'no2',
    'so2',
    'co',
    'temperature',
    'rain',
    'wind_direction',
    'wind_speed',
    'humidity',
    'pressure',
    'dew_point'
]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("Missing expected columns:", missing_columns)
print("Unexpected columns:", extra_columns)

Missing expected columns: []
Unexpected columns: []


In [7]:
df.dtypes

pm10              float64
pm25              float64
no2               float64
so2               float64
co                  int64
temperature       float64
rain              float64
wind_direction      int64
wind_speed        float64
humidity            int64
pressure          float64
dew_point         float64
dtype: object

In [8]:
non_numeric = df.select_dtypes(
    exclude=np.number
).columns.tolist()

print(
    "Non-numeric analytical columns:",
    non_numeric
)

Non-numeric analytical columns: []


In [9]:
print(
    "NaN values:",
    df.isna().sum().sum()
)

print(
    "Positive infinity:",
    np.isposinf(df.to_numpy()).sum()
)

print(
    "Negative infinity:",
    np.isneginf(df.to_numpy()).sum()
)

NaN values: 0
Positive infinity: 0
Negative infinity: 0


In [10]:
#variable groups
pollutants = [
    'pm25',
    'pm10',
    'no2',
    'so2',
    'co'
]

weather_variables = [
    'temperature',
    'rain',
    'wind_direction',
    'wind_speed',
    'humidity',
    'pressure',
    'dew_point'
]

In [11]:
#descriptive quality table
quality_summary = pd.DataFrame({
    'count': df.count(),
    'missing': df.isna().sum(),
    'unique': df.nunique(),
    'mean': df.mean(),
    'median': df.median(),
    'std': df.std(),
    'min': df.min(),
    'q1': df.quantile(0.25),
    'q3': df.quantile(0.75),
    'max': df.max()
})

quality_summary

,count,missing,unique,mean,median,std,min,q1,q3,max
pm10,23352,0,918,48.610517,42.3,28.368394,0.3,27.4,64.5,221.2
pm25,23352,0,871,35.045422,29.0,23.252260,0.2,18.8,44.6,170.4
no2,23352,0,763,14.271446,10.6,13.574662,0.0,3.7,20.7,107.3
so2,23352,0,335,6.767750,5.6,4.767855,0.0,3.7,8.3,37.5
co,23352,0,2206,669.837359,523.0,469.375936,59.0,333.0,839.0,3204.0
temperature,23352,0,307,17.811986,18.7,5.825566,1.3,13.3,22.0,32.6
rain,23352,0,137,0.224289,0.0,1.041249,0.0,0.0,0.0,23.1
wind_direction,23352,0,343,203.802073,175.0,87.014629,2.0,165.0,297.0,360.0
wind_speed,23352,0,159,4.106539,3.7,2.313208,0.0,2.5,5.3,18.8
humidity,23352,0,93,73.341384,77.0,20.505056,7.0,60.0,91.0,100.0


In [12]:
#negative pollutant concentrations
negative_pollutants = {}

for pollutant in pollutants:

    count = (
        df[pollutant] < 0
    ).sum()

    negative_pollutants[pollutant] = count

pd.Series(
    negative_pollutants,
    name='negative_values'
)

pm25    0
pm10    0
no2     0
so2     0
co      0
Name: negative_values, dtype: int64

In [13]:
weather_checks = {
    'humidity_below_0':
        (df['humidity'] < 0).sum(),

    'humidity_above_100':
        (df['humidity'] > 100).sum(),

    'negative_rain':
        (df['rain'] < 0).sum(),

    'negative_wind_speed':
        (df['wind_speed'] < 0).sum(),

    'wind_direction_below_0':
        (df['wind_direction'] < 0).sum(),

    'wind_direction_above_360':
        (df['wind_direction'] > 360).sum(),

    'nonpositive_pressure':
        (df['pressure'] <= 0).sum()
}

pd.Series(weather_checks)

humidity_below_0            0
humidity_above_100          0
negative_rain               0
negative_wind_speed         0
wind_direction_below_0      0
wind_direction_above_360    0
nonpositive_pressure        0
dtype: int64

In [14]:
#Dew point consistency
dew_above_temp = (
    df['dew_point'] > df['temperature']
)

print(
    "Dew point > temperature:",
    dew_above_temp.sum()
)

Dew point > temperature: 0


In [15]:
zero_counts = pd.DataFrame({
    'zero_count':
        (df == 0).sum(),

    'zero_percent':
        (df == 0).mean() * 100
})

zero_counts.sort_values(
    'zero_percent',
    ascending=False
)

,zero_count,zero_percent
rain,18325,78.472936
no2,423,1.811408
wind_speed,47,0.201268
dew_point,24,0.102775
so2,2,0.008565
pm10,0,0.000000
pm25,0,0.000000
co,0,0.000000
temperature,0,0.000000
wind_direction,0,0.000000
